In [1]:
import torch
import torchvision
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision.transforms
import torch.utils.data as dataloader

In [2]:
transform_train = torchvision.transforms.Compose([
    torchvision.transforms.RandomCrop(32, padding=4),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_val = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = torchvision.datasets.CIFAR10(root = './data', train = True, download = True, transform = transform_train)
val_dataset = torchvision.datasets.CIFAR10(root = './data', train = False, download = True, transform = transform_val)

KeyboardInterrupt: 

In [ ]:
num_classes = 10
batch_size = 128
num_channels = 3
img_size = 32
patch_size = 4
num_patches = (img_size // patch_size) ** 2
print(f"Number of patches: {num_patches}")
embedding_dim = 64
attention_heads = 4
transformer_blocks = 4
mlp_hidden_nodes = 128
learning_rate = 5e-4

Number of patches: 64


In [ ]:
train_loader = dataloader.DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_loader = dataloader.DataLoader(val_dataset, batch_size = batch_size, shuffle = False)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self):
        super().__init__()

        self.path_embed = nn.Conv2d(num_channels, embedding_dim, patch_size, patch_size)

    def forward(self, x):
        x = self.path_embed(x)
        x = x.flatten(2).transpose(1, 2) # To Faltten the path embedding and covert it to 64,16,64 by taking the transpose of the 1 , 2 position
        return x

In [ ]:
class Transformer_encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)

        # Set batch_first=True to resolve batch/sequence dimension ordering bug
        self.multihead_attention = nn.MultiheadAttention(embedding_dim, num_heads = attention_heads, batch_first=True)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, mlp_hidden_nodes),
            nn.GELU(),
            nn.Linear(mlp_hidden_nodes, embedding_dim)
        )

    def forward(self, x):
        residual1 = x
        x = self.layer_norm1(x)
        x = self.multihead_attention(x, x, x)[0]

        x = residual1 + x

        residual2 = x
        x = self.layer_norm2(x)
        x = self.mlp(x)

        x = residual2 + x

        return x

In [ ]:
class MLP_head(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.mlp_head = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        x = self.layer_norm1(x)
        x = self.mlp_head(x)

        return x

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.patch_embedding = PatchEmbedding()

        # Initialize tokens as zeros first, will be initialized in _init_weights
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embedding_dim))
        self.position_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, embedding_dim))

        # Reduced dropout rate for more stable learning
        self.dropout = nn.Dropout(0.1)

        self.transformer_encoder = nn.Sequential(*[Transformer_encoder() for _ in range(transformer_blocks)])

        self.mlp_head = MLP_head()

        # Apply proper weights initialization
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        
        # Initialize cls_token and position_embedding with standard deviation 0.02
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.position_embedding, std=0.02)

    def forward(self, x):
        x = self.patch_embedding(x)

        batch_size, sequence_length, _ = x.size()

        cls_token = self.cls_token.expand(batch_size, -1, -1)

        x = torch.cat((cls_token, x), dim = 1)

        x = x + self.position_embedding

        x = self.dropout(x)

        x = self.transformer_encoder(x)

        cls_token_output = x[:, 0]

        output = self.mlp_head(cls_token_output)

        return output

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VisionTransformer().to(device)

# Switch to AdamW with decoupled weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate, weight_decay = 1e-2)
criterion = nn.CrossEntropyLoss()


In [ ]:
# Number of epochs
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0.0
    correct_epoch = 0
    total_epoch = 0

    print(f"\nEpoch {epoch + 1}/{epochs}")
    print("-" * 50)

    for batch_idx, (images, labels) in enumerate(train_loader):

        # Move data to GPU/CPU
        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Track loss
        total_loss += loss.item()

        # Predictions
        preds = outputs.argmax(dim=1)

        # Correct predictions in batch
        correct = (preds == labels).sum().item()

        # Batch accuracy
        accuracy = 100.0 * correct / labels.size(0)

        # Epoch statistics
        correct_epoch += correct
        total_epoch += labels.size(0)

        # Print every 100 batches
        if batch_idx % 100 == 0:
            print(
                f"Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Accuracy: {accuracy:.2f}%"
            )

    # Epoch metrics
    avg_loss = total_loss / len(train_loader)
    epoch_accuracy = 100.0 * correct_epoch / total_epoch

    print("\nEpoch Summary")
    print(
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )



Epoch 1/5
--------------------------------------------------


RuntimeError: Given groups=1, weight of size [64, 3, 4, 4], expected input[128, 1, 28, 28] to have 3 channels, but got 1 channels instead